In [19]:
from fem import (
    GMSHtools,
    FEMModel,
    plot_gmsh_mesh,
    globalParameters,
)
import os
import numpy as np
import matplotlib.pyplot as plt
import gmsh
np.set_printoptions(suppress=True, precision=6, linewidth=400)


In [20]:
globalParameters['nDoF'] = 3
globalParameters['nDIM'] = 3

In [21]:
# General model parameters
output_file = 'example_PP1.msh'

In [22]:
# read mesh — node_map and system_nDof auto-generated
mesh = GMSHtools(output_file)

  MESH SUMMARY

  === NODES ===  (6834 total — showing first 3)
     Tag              x              y              z
--------------------------------------------------------------------------------
       1         0.0000         0.0000         0.0000
       2      5000.0000         0.0000         0.0000
       3      5000.0000       300.0000         0.0000
--------------------------------------------------------------------------------

  === PHYSICAL GROUPS ===  (4 total)
      ID    Dim   Name
--------------------------------------------------------------------------------
      21      1   'support_xy'
      22      1   'support_y'
      23      2   'load'
      24      3   'solid'
--------------------------------------------------------------------------------

  === ELEMENTS ===  (4 groups)
      ID    Dim     Type   Nodes/el   N elements   Name
--------------------------------------------------------------------------------
      21      1        1          2            6   'su

In [23]:
# plot_gmsh_mesh(mesh,
#                show_node_labels    = False,
#                show_element_labels = False,
#                show_node_points    = False,
#                view_3d             = True, elev=45, azim=-45,
#                figsize             = (12, 8))

In [24]:

load_dictionary = {
                23:   {'value': 1, 'direction': '-z'},     
}

In [25]:
model = FEMModel(
    mesh                = mesh,
    section_dictionary  = {},
    restrain_dictionary = {},
    load_dictionary     = load_dictionary,
    element_class_map   = None,       # OpenSees only
    analysis_type       = '3D',
    consistent_loads    = False,
)


  FEM MODEL SUMMARY
--------------------------------------------------------------------------------
  Analysis type    : 3D
  Nodes            : 6834
  system_nDof      : 20502
  Elements         : None (OpenSees)
  Steps FEM        : 0
  Steps OpenSees   : 0

  --- Sections ---

  --- Restrained nodes ---

  --- Loaded nodes (dim=0) ---

  --- Load vector ---
  Non-zero DOFs in F_load : 811
  Total applied force     : -746604.5993 (x)  -753395.4007 (y)
--------------------------------------------------------------------------------



In [26]:
# Mesh diagnostics
model.check_mesh()


  MESH DIAGNOSTICS
--------------------------------------------------------------------------------
  Nodes            : 6834
  system_nDof      : 20502
  Elements         : None (OpenSees)
  Physical groups  : 4

  --- Orphan nodes ---
  OK — no orphan nodes

  --- Physical groups ---
      ID   Dim  Name                    Elements     Nodes  Section
  ------  ----  --------------------  ----------  --------  ----------
      21     1  support_xy                     6         7  N/A
      22     1  support_y                      6         7  N/A
      23     2  load                        1408       811  MISSING
      24     3  solid                      29336      6834  MISSING

  --- Restrained nodes ---
     Tag             x             y  Condition
  ------  ------------  ------------  ------------

  --- Load summary ---
  Non-zero DOFs    : 811
  Total Fx         : -746604.5993
  Total Fy         : -753395.4007
-----------------------------------------------------------------

## Opensees

In [27]:
import openseespy.opensees as ops
import opsvis as opsv

ops.wipe()
ops.model('basicBuilder', '-ndm', 3, '-ndf', 3)


In [28]:
# Nodes
for tag, (x, y, z) in mesh.nodes.items():
    ops.node(tag, x, y , z)

In [29]:
# Boundary conditions
fixed_nodes = set()
for tag in mesh.physical_groups['support_xy'].nodes:
    if tag not in fixed_nodes:
        fixed_nodes.add(tag)
        # ops.fix(tag, 1, 1, 1, 1, 1, 1)
        ops.fix(tag, 1, 1, 1)

In [30]:
# Boundary conditions
fixed_nodes = set()
for tag in mesh.physical_groups['support_y'].nodes:
    if tag not in fixed_nodes:
        fixed_nodes.add(tag)
        # ops.fix(tag, 1, 1, 1, 1, 1, 1)
        ops.fix(tag, 0, 1, 1)

In [31]:
# Material
E = 3500      
nu = 0.36     
rho = 2400e-9  
ops.nDMaterial('ElasticIsotropic', 1, E, nu, rho)

In [32]:
# Elements
group = mesh.physical_groups['solid'].elements
for elem_tag, conn in zip(group['element_tags'], group['connectivity']):
    n1, n2, n3, n4 = conn
    ops.element('FourNodeTetrahedron', elem_tag, n1, n2, n3, n4, 1)

In [33]:
ops.timeSeries('Linear', 1)
ops.pattern('Plain', 1, 1)

for tag, force in model.F_nodal.items():
    if np.any(np.abs(force) > 0):
        ops.load(tag, *force.tolist())

In [34]:
NstepGravity = 10
DGravity     = 1 / NstepGravity

ops.system("SparseSYM")
ops.numberer("RCM") 
ops.constraints("Plain")
ops.integrator("LoadControl", DGravity)
ops.test("NormUnbalance", 1.0e-6, 100, 0)
ops.algorithm("Newton")
ops.analysis("Static")

for step in range(NstepGravity):
    ops.analyze(1)
    model.set_results_opensees(ops, step=step)

# ops.analyze(NstepGravity)
# # save last step
# model.set_results_opensees(ops, step=0, time=1.0)

In [35]:
# Static results in gmsh — last step
model.plot2gmsh(
    step           = -1,
    source         = 'opensees',
    disp_factor    = 2,
    show_disp      = True,
    show_loads     = True,
    show_reactions = True,
    show_stress    = True,
    show_strain    = True,
    show_vm        = True,
    show_averaged  = True,
)

In [36]:
# Animate displacements in gmsh
model.plot2gmsh_animate(disp_factor=2)